<a href="https://colab.research.google.com/github/Leonanda1013/DataLoverz/blob/main/Competition/GammaFest_IPB/3_Model_LighGBM_Poisson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.multioutput import MultiOutputRegressor

from lightgbm import LGBMRegressor

In [15]:
df_train = pd.read_csv('/content/train_clean.csv')
df_test = pd.read_csv('/content/test_clean.csv')

In [16]:
train = df_train.copy()
test = df_test.copy()

test_id = test['Id']

train = train.drop(columns=['id'], errors='ignore')
test = test.drop(columns=['id'], errors='ignore')

In [17]:
target_cols = ['team_goals', 'opp_goals']

X = train.drop(columns=target_cols)
y = train[target_cols]

In [18]:
cat_cols = X.select_dtypes(include=['object', 'category']).columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns

In [19]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

In [20]:
base_model = LGBMRegressor(
    objective='poisson',   # 🔥 penting!
    n_estimators=500,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model = MultiOutputRegressor(base_model)

In [21]:
pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', model)
])

In [22]:
pipeline.fit(X, y)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015961 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2215
[LightGBM] [Info] Number of data points in the train set: 78772, number of used features: 37
[LightGBM] [Info] Start training from score 0.446683
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013497 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2215
[LightGBM] [Info] Number of data points in the train set: 78772, number of used features: 37
[LightGBM] [Info] Start training from score 0.446683


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  Index(['gender', 'is_home', 'neutral', 'tournament', 'population_team',
       'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp',
       'altitude_venue', 'distance_travel_team', 'distance_travel_opp',
       'temperature_venue', 'year', 'month', 'match_era',
       'population_team_missing', 'p...
       'confederation_opp_CONMEBOL', 'confederation_opp_OFC',
       'confederation_opp_UEFA', 'confederation_opp_Unknown'],
      dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index([], dtype='object'))])),
                ('model',
                 MultiOutputRegressor(estimator=LGBMRegressor(colsample_bytree=0.8,
                                                              learning_rate=0.05,
                                                              n_estimators=500,
                                                              objective='poisson',
                                                              random_state=42,
                                                              subsample=0.8)))])

In [23]:
preds = pipeline.predict(test)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [24]:
submission = pd.DataFrame({
    'Id': test_id,
    'team_goals': np.round(preds[:, 0]).astype(int),
    'opp_goals': np.round(preds[:, 1]).astype(int)
})

In [25]:
submission.to_csv('submission_gmc_LighGBM_Poisson.csv', index = False)

In [26]:
submission.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42422 entries, 0 to 42421
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Id          42422 non-null  object
 1   team_goals  42422 non-null  int64 
 2   opp_goals   42422 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 994.4+ KB
